# AttentionEncoder — sanity test for `FeatureAttentionEncoder` + KAN

Goal: verify the **attention encoder feeding the KAN** (`AttentionKANModel`)
trains and saves correctly, before running the full multi-seed comparison.

This notebook mirrors `P1_structurelevel/KL-h_reliablescore.ipynb` (the proven
KAN + LTN pipeline: 18 features, 6 classes, satisfaction loss). The **only**
architectural change is:

```
MultiKANModel(kan)   ->   AttentionKANModel(IN, MultiKANModel(kan), ...)
```

so the attention encoder refines the 18 features (same-dim output) and feeds
the KAN. Everything else — data, LTN rules, loss — is identical, so any change
in behaviour is attributable to the encoder.

To switch to the 4-class / 9-subclass (13-output) setup later, edit only the
**Config** cell (DATA_PATH, mappings, IN_FEATURES, N_CLASSES, KAN width).


In [1]:
# --- imports & paths ---
import os, sys
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler

# attention_modules.py lives in this folder (P5_attention);
# utils.py / MultiKANModel live in P1_structurelevel.
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('../P1_structurelevel'))

from utils import LogitsToPredicate, MultiKANModel, DataLoader, DataLoaderMulti
from attention_modules import AttentionKANModel, FeatureAttentionEncoder
from kan import KAN
import ltn
import ltn.fuzzy_ops

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


In [ ]:
# --- Config (edit here to change dataset / setup) ---
# These files already contain pre-scaled numeric features + an integer
# label_L2 (0-5). There is NO label_L1 column -> we derive it (see data cell).
TRAIN_PATH = '../P1_structurelevel/efficiency/input_files/logiKNet_train_10000.csv'
TEST_PATH  = '../P1_structurelevel/efficiency/input_files/logiKNet_test_3994.csv'

X_columns = [
    'Header_Length', 'Protocol Type', 'Duration', 'Rate', 'Srate',
    'IPv', 'LLC',
    'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number',
    'Magnitue', 'Radius', 'Covariance',
]

# label_L2 is already an integer index 0..5 in these files. Names for reference:
label_L2_names = {
    0: "MQTT-DDoS-Connect_Flood", 1: "MQTT-DDoS-Publish_Flood",
    2: "MQTT-DoS-Connect_Flood", 3: "MQTT-DoS-Publish_Flood",
    4: "MQTT-Malformed_Data", 5: "Benign",
}
# hierarchy: classes 0..4 -> MQTT (L1=0), class 5 -> Benign (L1=1)
BENIGN_L2 = 5

IN_FEATURES = len(X_columns)      # 18
N_CLASSES   = len(label_L2_names) # 6 (L2 target)
KAN_WIDTH   = [IN_FEATURES, 6, 6, N_CLASSES]

# attention encoder hyperparameters
ATTN = dict(d_model=32, n_heads=4, n_layers=2, dropout=0.1, residual=True)

SEED   = 42
EPOCHS = 2000   # safety cap only; early stopping normally ends training first
LR     = 1e-3

# FAST_DEV: on CPU, train on the small 3994-row file (the "test" file) just to
# confirm the encoder+KAN learns/runs. Evaluation is done once, at the END.
# Set FALSE for the real run (train on logiKNet_train_35945.csv).
FAST_DEV  = False
EVAL_EVERY = 25   # only print loss/timing every N epochs during training

SAVE_DIR = './saved'
os.makedirs(SAVE_DIR, exist_ok=True)
print('IN_FEATURES =', IN_FEATURES, '| N_CLASSES =', N_CLASSES, '| KAN_WIDTH =', KAN_WIDTH)


IN_FEATURES = 18 | N_CLASSES = 6 | KAN_WIDTH = [18, 6, 6, 6]


In [ ]:
# --- 1. Read input data ---
# Separate train/test files; label_L2 is already integer 0..5 (do NOT map).
if FAST_DEV:
    # quick CPU sanity run: small file as train, evaluate on the same set at end
    train_df = pd.read_csv(TEST_PATH)
    test_df  = pd.read_csv(TEST_PATH)
    print('FAST_DEV: training on the 3994-row file (sanity check only)')
else:
    train_df = pd.read_csv(TRAIN_PATH)
    test_df  = pd.read_csv(TEST_PATH)

# sanity: required columns present
missing = [c for c in X_columns + ['label_L2'] if c not in train_df.columns]
assert not missing, f'missing columns in train file: {missing}'

# derive label_L1 from label_L2 (0..4 -> MQTT=0, 5 -> Benign=1)
for d in (train_df, test_df):
    d['label_L1'] = (d['label_L2'] == BENIGN_L2).astype(int)

scaler = StandardScaler()
train_X_scaled = scaler.fit_transform(train_df[X_columns])
test_X_scaled  = scaler.transform(test_df[X_columns])
print('train shape:', train_X_scaled.shape, '| test shape:', test_X_scaled.shape)
print('NaN/Inf in train:', np.isnan(train_X_scaled).any(), np.isinf(train_X_scaled).any())
print('NaN/Inf in test :', np.isnan(test_X_scaled).any(), np.isinf(test_X_scaled).any())

train_y = train_df[['label_L1', 'label_L2']].values
test_y  = test_df[['label_L1', 'label_L2']].values
print('train L2 dist:', np.bincount(train_y[:, 1]))
print('test  L2 dist:', np.bincount(test_y[:, 1]))

# full-batch loaders (matches the reference notebook). Target = label_L2.
train_loader = DataLoader(
    data=torch.tensor(train_X_scaled, dtype=torch.float32, device=device),
    labels=torch.tensor(train_y[:, 1], dtype=torch.long, device=device),
    batch_size=len(train_df))
test_loader = DataLoader(
    data=torch.tensor(test_X_scaled, dtype=torch.float32, device=device),
    labels=torch.tensor(test_y[:, 1], dtype=torch.long, device=device),
    batch_size=len(test_df))


In [ ]:
# --- 2. LTN setup (connectives, quantifiers, class constants) ---
Not    = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And    = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or     = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
SatAgg = ltn.fuzzy_ops.SatAgg()

l_MQTT_DDoS_Connect_Flood = ltn.Constant(torch.tensor([1, 0, 0, 0, 0, 0]))
l_MQTT_DDoS_Publish_Flood = ltn.Constant(torch.tensor([0, 1, 0, 0, 0, 0]))
l_MQTT_DoS_Connect_Flood  = ltn.Constant(torch.tensor([0, 0, 1, 0, 0, 0]))
l_MQTT_DoS_Publish_Flood  = ltn.Constant(torch.tensor([0, 0, 0, 1, 0, 0]))
l_MQTT_Malformed_Data     = ltn.Constant(torch.tensor([0, 0, 0, 0, 1, 0]))
l_Benign                  = ltn.Constant(torch.tensor([0, 0, 0, 0, 0, 1]))


In [ ]:
# --- helpers: satisfaction + accuracy ---
def compute_sat_levels(loader, P):
    sat_level = 0
    for data, labels in loader:
        x_c0 = ltn.Variable("x_c0", data[labels == 0])
        x_c1 = ltn.Variable("x_c1", data[labels == 1])
        x_c2 = ltn.Variable("x_c2", data[labels == 2])
        x_c3 = ltn.Variable("x_c3", data[labels == 3])
        x_c4 = ltn.Variable("x_c4", data[labels == 4])
        x_c5 = ltn.Variable("x_c5", data[labels == 5])
        x_MQTT = ltn.Variable("x_MQTT", data[labels < 5])   # hierarchy: MQTT umbrella
        sat_level = SatAgg(
            Forall(x_c0, P(x_c0, l_MQTT_DDoS_Connect_Flood)),
            Forall(x_c1, P(x_c1, l_MQTT_DDoS_Publish_Flood)),
            Forall(x_c2, P(x_c2, l_MQTT_DoS_Connect_Flood)),
            Forall(x_c3, P(x_c3, l_MQTT_DoS_Publish_Flood)),
            Forall(x_c4, P(x_c4, l_MQTT_Malformed_Data)),
            Forall(x_c5, P(x_c5, l_Benign)),
            Forall(x_MQTT, Not(P(x_MQTT, l_Benign))),
        )
    return sat_level


def compute_accuracy(loader, model):
    total_correct, total = 0, 0
    model.eval()
    with torch.no_grad():
        for data, labels in loader:
            logits = model(data)
            preds = torch.argmax(logits, dim=1)
            total_correct += (preds == labels).sum()
            total += labels.numel()
    return (total_correct.float() / total).item()


def reliable_score(loader, model):
    score = 0
    model.eval()
    with torch.no_grad():
        for data, labels in loader:
            preds = torch.argmax(model(data), dim=1)
            for i in range(len(labels)):
                if labels[i] == preds[i]:
                    score += 1
                elif labels[i] < 5 and preds[i] < 5:   # both MQTT (L1 correct)
                    score += 0.5
    return score / loader.batch_size


In [ ]:
# --- 3. Build AttentionKANModel = FeatureAttentionEncoder -> KAN ---
torch.manual_seed(SEED)
np.random.seed(SEED)

kan = KAN(width=KAN_WIDTH, grid=5, k=3, seed=SEED, device=device)
model = AttentionKANModel(IN_FEATURES, MultiKANModel(kan), **ATTN).to(device)

# sanity: forward pass shape + attention map shape
with torch.no_grad():
    _logits = model(train_loader.data[:8])
print('logits shape:', tuple(_logits.shape), '(expect [8,', N_CLASSES, '])')
print('attention map shape:', tuple(model.last_attention().shape))

P = ltn.Predicate(LogitsToPredicate(model))
optimizer = torch.optim.Adam(P.parameters(), lr=LR)
print('trainable params:', sum(p.numel() for p in P.parameters() if p.requires_grad))


In [ ]:
# --- train with satisfaction loss; stop when loss converges ---
# Early stop when |loss change| < LOSS_TOL for PATIENCE consecutive epochs.
# EPOCHS stays as a safety cap. stop_epoch tells you the budget to set for
# the fixed-epoch multi-seed runs.
import time

LOSS_TOL = 1e-3
PATIENCE = 5

hist_loss = []
stop_epoch = EPOCHS - 1
small_count = 0
for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    optimizer.zero_grad()
    sat = compute_sat_levels(train_loader, P)   # training objective only
    loss = 1. - sat
    loss.backward()
    optimizer.step()
    step_t = time.time() - t0
    hist_loss.append(loss.item())

    if epoch % EVAL_EVERY == 0:
        print(f"epoch {epoch:4d} | loss {loss.item():.4f} "
              f"| train sat {sat.item():.4f} | step {step_t*1000:.0f} ms")

    if epoch > 0:
        delta = abs(hist_loss[-1] - hist_loss[-2])
        small_count = small_count + 1 if delta < LOSS_TOL else 0
        if small_count >= PATIENCE:
            stop_epoch = epoch
            print(f"converged: |dloss| < {LOSS_TOL} for {PATIENCE} epochs, "
                  f"stopping at epoch {epoch} (loss {loss.item():.4f})")
            break

print(f"\n>>> stopped at epoch {stop_epoch} "
      f"(use ~this many epochs for the fixed multi-seed runs)")

# --- final evaluation, ONCE, after training ---
print('\n=== final evaluation ===')
final_acc = compute_accuracy(test_loader, model)
final_sat = compute_sat_levels(test_loader, P)
final_rel = reliable_score(test_loader, model)
print(f"test accuracy : {final_acc:.4f}")
print(f"test sat      : {final_sat.item():.4f}")
print(f"reliable score: {final_rel:.4f}")


In [ ]:
# --- quick look at the learned feature-feature attention (interpretability) ---
import matplotlib.pyplot as plt
with torch.no_grad():
    _ = model(test_loader.data)
A = model.last_attention().mean(0).cpu().numpy()   # [F, F] averaged over test
plt.figure(figsize=(6, 5))
plt.imshow(A, cmap='viridis')
plt.colorbar(label='attention weight')
plt.title('Mean feature-feature attention')
plt.xlabel('key feature'); plt.ylabel('query feature')
plt.xticks(range(IN_FEATURES), X_columns, rotation=90, fontsize=6)
plt.yticks(range(IN_FEATURES), X_columns, fontsize=6)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'attention_map.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- 4. Save the model for evaluation ---
# Save state_dict + everything needed to rebuild for eval (robust for KAN).
ckpt_path = os.path.join(SAVE_DIR, 'attn_kan_2_6.pt')
torch.save({
    'model_state': model.state_dict(),
    'config': {
        'IN_FEATURES': IN_FEATURES, 'N_CLASSES': N_CLASSES,
        'KAN_WIDTH': KAN_WIDTH, 'ATTN': ATTN, 'SEED': SEED,
        'X_columns': X_columns,
        'label_L2_names': label_L2_names,
        'BENIGN_L2': BENIGN_L2,
    },
    'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_,
    'history': {'loss': hist_loss},
    'final': {'acc': final_acc, 'sat': final_sat.item(), 'reliable': final_rel},
}, ckpt_path)
print('saved checkpoint ->', ckpt_path)

# NOTE: we do NOT torch.save(model, ...) the whole module. pykan's
# Symbolic_KANLayer stores lambda functions, which Python cannot pickle
# (AttributeError: Can't get local object '...<lambda>'). The state_dict
# checkpoint above is the portable path -- rebuild + load_state_dict to reload
# (see the "Reload for evaluation" cell below).


## Reload for evaluation

```python
import torch
from utils import MultiKANModel
from attention_modules import AttentionKANModel
from kan import KAN

ckpt = torch.load('./saved/attn_kan_2_6.pt', map_location=device)
cfg = ckpt['config']
kan = KAN(width=cfg['KAN_WIDTH'], grid=5, k=3, seed=cfg['SEED'], device=device)
model = AttentionKANModel(cfg['IN_FEATURES'], MultiKANModel(kan), **cfg['ATTN']).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

# re-apply the saved scaler to new data before inference:
#   X = (X_raw - ckpt['scaler_mean']) / ckpt['scaler_scale']
```

Then feed it to `eval_metrics.evaluate_run(model, test_loader, n_classes=cfg['N_CLASSES'], device=device)`.

### Switching to the 4-class / 9-subclass (13-output) setup
Edit only the **Config** cell: point `DATA_PATH` at `filtered_train_s_4_9.csv`,
use the 4-way `label_L1_mapping` and 9-way `label_L2_mapping`, set
`N_CLASSES = 13` (or your L2 count) and `KAN_WIDTH` accordingly, then update the
LTN constants/rules in the *LTN setup* and *compute_sat_levels* cells to match
the hierarchical rules in `KAN2LTN+hierarchy.py`.
